# Cell tracking benchmark: random walk vs. persistent random walk

Cells move on a **40 x 40 grid** over **41 frames** (5 min each = 200 min total).
We compare how well different tracking methods recover the *true* trajectories,
for **two motion models** and **three cell densities**, and we estimate the
**critical difficulty ratio** at which each method starts to fail.

**Motion models (all cells share the same model in a given run)**

| Model | Description |
|---|---|
| `random_walk` | Brownian motion — each step is an independent Gaussian displacement (no memory). |
| `persistent`  | Persistent / correlated random walk — a cell keeps a heading that drifts slowly, so it moves in fairly straight runs. |

Both models are tuned to the **same RMS per-frame displacement**, so the
*difficulty ratio* `r = displacement / neighbour-spacing` is defined identically
and the two models are directly comparable.

**Methods**: greedy nearest-neighbour, Hungarian/LAP (`scipy`),
LapTrack (`laptrack`), trackpy (`trackpy`). Optional packages are skipped if absent.

**Design**: densities **n = 5, 48, 480**, **N = 15 replications** per
(motion, density, method). Ground truth is known, so we score the links each
tracker produces (precision / recall / F1 + track purity).


In [ ]:
# Optional installs (uncomment if needed)
# %pip install numpy pandas scipy matplotlib
# %pip install laptrack trackpy

## 1. Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

# ---- Simulation configuration ----
GRID        = 40.0     # 40 x 40 grid (arbitrary units, e.g. microns)
N_FRAMES    = 41       # number of time frames
DT_MIN      = 5        # minutes between frames
SIGMA_STEP  = 1.5      # random-walk step std -> sets the displacement scale
SIGMA_THETA = 0.35     # persistent walk: heading angular noise (rad/frame).
                       #   small -> very straight; large -> ~diffusive
SIGMA_LOC   = 0.10     # localization (detection) noise std
MAX_DIST    = 6.0      # linking search radius / gating (shared by all methods)

DENSITIES   = [5, 48, 480]                    # number of cells on the grid
MOTIONS     = ["random_walk", "persistent"]   # the two motion models
N_REPEATS   = 15                              # replications per (motion, density, method)
BASE_SEED   = 20260706

rng_master = np.random.default_rng(BASE_SEED)


def nn_spacing(n, grid=GRID):
    '''Mean nearest-neighbour spacing for n points uniform on a grid x grid area:
    0.5 / sqrt(rho), rho = n / area.'''
    return 0.5 / np.sqrt(n / (grid * grid))


def rms_displacement(sigma=SIGMA_STEP):
    '''RMS per-frame displacement (both axes) of the Gaussian step.'''
    return sigma * np.sqrt(2.0)


def difficulty_ratio(n, sigma=SIGMA_STEP, grid=GRID):
    '''Difficulty knob = per-frame displacement / neighbour spacing.'''
    return rms_displacement(sigma) / nn_spacing(n, grid)


print(f"Grid {GRID:.0f}x{GRID:.0f} | {N_FRAMES} frames x {DT_MIN} min "
      f"= {(N_FRAMES-1)*DT_MIN} min | RMS displacement = {rms_displacement():.2f}")
print(f"{'density n':>10} | {'NN spacing':>10} | {'ratio r':>8}")
print("-" * 34)
for n in DENSITIES:
    print(f"{n:>10d} | {nn_spacing(n):>10.2f} | {difficulty_ratio(n):>8.3f}")

## 2. Simulation: two motion models

Both models are matched to the same per-frame displacement scale
(`step = sigma*sqrt(2)`), so only the *character* of the motion differs:

* **random walk** — independent Gaussian jumps (no directional memory).
* **persistent** — fixed step length along a heading that random-walks by
  `SIGMA_THETA` per frame; the directional autocorrelation (persistence) is
  `exp(-SIGMA_THETA**2 / 2)` per frame (≈ 0.94 here).

`labels[t][i]` gives the true cell id of detection `i` in frame `t`; detections
are order-shuffled and carry localization noise so trackers can't exploit input
order.

In [ ]:
def _reflect(p, grid=GRID):
    '''Reflect coordinates back into [0, grid].'''
    p = np.abs(p)
    p = grid - np.abs(grid - p)
    return np.clip(p, 0, grid)


def simulate_random_walk(n_cells, n_frames=N_FRAMES, grid=GRID,
                         sigma=SIGMA_STEP, seed=None):
    '''Brownian motion: independent Gaussian steps.'''
    rng = np.random.default_rng(seed)
    pos = np.empty((n_frames, n_cells, 2))
    pos[0] = rng.uniform(0, grid, size=(n_cells, 2))
    for t in range(1, n_frames):
        pos[t] = _reflect(pos[t - 1] + rng.normal(0, sigma, size=(n_cells, 2)), grid)
    return pos


def simulate_persistent(n_cells, n_frames=N_FRAMES, grid=GRID,
                        sigma=SIGMA_STEP, sigma_theta=SIGMA_THETA, seed=None):
    '''Persistent random walk: fixed step length along a slowly drifting heading.
    Step length = sigma*sqrt(2) to match the random walk's RMS displacement.'''
    rng = np.random.default_rng(seed)
    step = sigma * np.sqrt(2.0)
    pos = np.empty((n_frames, n_cells, 2))
    pos[0] = rng.uniform(0, grid, size=(n_cells, 2))
    theta = rng.uniform(0, 2 * np.pi, size=n_cells)
    for t in range(1, n_frames):
        theta = theta + rng.normal(0, sigma_theta, size=n_cells)
        disp = np.column_stack([step * np.cos(theta), step * np.sin(theta)])
        cand = pos[t - 1] + disp
        # reflect heading off walls so cells turn back into the field
        hit_x = (cand[:, 0] < 0) | (cand[:, 0] > grid)
        hit_y = (cand[:, 1] < 0) | (cand[:, 1] > grid)
        theta[hit_x] = np.pi - theta[hit_x]
        theta[hit_y] = -theta[hit_y]
        pos[t] = _reflect(cand, grid)
    return pos


def simulate_cells(n_cells, motion, seed=None):
    if motion == "random_walk":
        return simulate_random_walk(n_cells, seed=seed)
    elif motion == "persistent":
        return simulate_persistent(n_cells, seed=seed)
    raise ValueError(motion)


def make_detections(pos, sigma_loc=SIGMA_LOC, seed=None):
    '''Add localization noise and shuffle detection order within each frame.'''
    rng = np.random.default_rng(seed)
    n_frames, n_cells, _ = pos.shape
    dets, labels = [], []
    for t in range(n_frames):
        order = rng.permutation(n_cells)
        p = pos[t, order] + rng.normal(0, sigma_loc, size=(n_cells, 2))
        dets.append(p)
        labels.append(order.copy())
    return dets, labels


# sanity check
for mtn in MOTIONS:
    _p = simulate_cells(8, mtn, seed=0)
    disp = np.linalg.norm(np.diff(_p, axis=0), axis=2).mean()
    print(f"{mtn:>12}: mean per-frame displacement = {disp:.2f}")

## 2b. Motion-model check: MSD scaling exponent alpha (RW vs PRW)

The **mean squared displacement** as a function of lag time tau follows a power
law `MSD(tau) ~ tau^alpha`, so on a **log MSD vs log tau** plot the slope is the
anomalous-diffusion exponent **alpha**:

| regime | alpha | meaning |
|---|---|---|
| diffusive (Brownian / random walk) | ~ 1 | slope 1 across all lags |
| superdiffusive (persistent walk, short lags) | 1 < alpha <= 2 | ballistic runs; alpha -> 2 for perfectly straight |
| crossover (persistent walk, long lags) | -> 1 | direction memory lost -> looks diffusive again |

So alpha directly tells you whether a trajectory *behaves* like RW or PRW. We
characterize this on **free (unbounded)** versions of the same step rules, so the
power law is clean (the 40x40 reflecting walls used in the tracking experiment
would cap MSD at long lags, bending the curve). The short-lag alpha is identical
either way.

**Is this the *true* alpha?** Yes — and to prove it we overlay the **exact
analytical MSD** for each model and fit alpha from that too:

* random walk: `MSD(tau) = 2*sigma^2 * tau`  ->  **alpha = 1 exactly**.
* persistent walk (discrete Furth formula): with step `s = sigma*sqrt(2)` and
  per-frame directional persistence `p = exp(-sigma_theta^2/2)`,
  `MSD(tau) = s^2 * [tau + 2 * sum_{k=1}^{tau-1} (tau-k) * p^k]`,
  which gives **alpha -> 2** as tau -> 0 and **alpha -> 1** as tau -> inf.

The table compares the *fitted* alpha (from simulated tracks) with the *true*
alpha (from theory) in each lag window; they should agree within noise.

In [ ]:
def free_random_walk(n_cells, n_frames, sigma=SIGMA_STEP, seed=None):
    '''Unbounded Brownian motion (cumulative Gaussian steps).'''
    rng = np.random.default_rng(seed)
    steps = rng.normal(0, sigma, size=(n_frames, n_cells, 2))
    steps[0] = 0.0
    return np.cumsum(steps, axis=0)


def free_persistent(n_cells, n_frames, sigma=SIGMA_STEP,
                    sigma_theta=SIGMA_THETA, seed=None):
    '''Unbounded persistent random walk (fixed-length steps, drifting heading).'''
    rng = np.random.default_rng(seed)
    step = sigma * np.sqrt(2.0)
    theta = rng.uniform(0, 2 * np.pi, size=n_cells)
    pos = np.zeros((n_frames, n_cells, 2))
    for t in range(1, n_frames):
        theta = theta + rng.normal(0, sigma_theta, size=n_cells)
        pos[t] = pos[t - 1] + np.column_stack([step * np.cos(theta),
                                               step * np.sin(theta)])
    return pos


def time_averaged_msd(pos, max_lag=None):
    '''Time- and ensemble-averaged MSD(tau). pos: (n_frames, n_cells, 2).'''
    n_frames = pos.shape[0]
    if max_lag is None:
        max_lag = n_frames - 1
    taus = np.arange(1, max_lag + 1)
    msd = np.array([np.mean(np.sum((pos[tau:] - pos[:-tau]) ** 2, axis=2))
                    for tau in taus])
    return taus, msd


def fit_alpha(taus, msd, lo, hi):
    '''Slope of log MSD vs log tau over lags [lo, hi] -> alpha.'''
    m = (taus >= lo) & (taus <= hi)
    slope, intercept = np.polyfit(np.log(taus[m]), np.log(msd[m]), 1)
    return slope, intercept


# ---- exact analytical (TRUE) MSD for each model ----
def msd_theory_rw(taus, sigma=SIGMA_STEP):
    '''MSD = 2 sigma^2 tau  ->  alpha = 1 exactly.'''
    return 2.0 * sigma ** 2 * np.asarray(taus, float)


def msd_theory_prw(taus, sigma=SIGMA_STEP, sigma_theta=SIGMA_THETA):
    '''Discrete Furth formula for a persistent random walk.'''
    s2 = 2.0 * sigma ** 2
    p = np.exp(-sigma_theta ** 2 / 2.0)          # per-frame directional persistence
    out = []
    for tau in np.asarray(taus, int):
        k = np.arange(1, tau)
        out.append(s2 * (tau + 2.0 * np.sum((tau - k) * p ** k)))
    return np.array(out)


MSD_THEORY = {"random_walk": msd_theory_rw, "persistent": msd_theory_prw}

# ---- characterize both models ----
MSD_CELLS   = 300      # cells to ensemble-average over (per-cell quantity)
MSD_FRAMES  = 200      # longer than the experiment for a clean power law
MSD_REPS    = 5
FIT_SHORT   = (1, 5)   # lag window for the short-time (ballistic) exponent
FIT_LONG    = (40, 150)  # lag window for the long-time exponent

free_sim = {"random_walk": free_random_walk, "persistent": free_persistent}
alpha_rows = []

plt.figure(figsize=(7.5, 6))
for motion, sim in free_sim.items():
    msds = []
    for rep in range(MSD_REPS):
        pos = sim(MSD_CELLS, MSD_FRAMES, seed=BASE_SEED + rep)
        taus, m = time_averaged_msd(pos, max_lag=MSD_FRAMES - 1)
        msds.append(m)
    msd = np.mean(msds, axis=0)
    theory = MSD_THEORY[motion](taus)

    a_s, _  = fit_alpha(taus, msd, *FIT_SHORT)       # fitted from simulation
    a_l, _  = fit_alpha(taus, msd, *FIT_LONG)
    ta_s, _ = fit_alpha(taus, theory, *FIT_SHORT)    # true, from analytical MSD
    ta_l, _ = fit_alpha(taus, theory, *FIT_LONG)
    alpha_rows.append({"motion": motion,
                       "alpha_short_sim": round(a_s, 2),  "alpha_short_true": round(ta_s, 2),
                       "alpha_long_sim":  round(a_l, 2),  "alpha_long_true":  round(ta_l, 2)})

    line, = plt.loglog(taus, msd, "o", ms=3,
                       label=f"{motion} sim (a_s={a_s:.2f}, a_l={a_l:.2f})")
    plt.loglog(taus, theory, "-", color=line.get_color(), lw=1.6,
               label=f"{motion} theory (a_s={ta_s:.2f}, a_l={ta_l:.2f})")

# reference slopes
t_ref = np.array([taus[0], taus[-1]])
plt.loglog(t_ref, msd[0] * (t_ref / t_ref[0]) ** 1, "k--", lw=1, alpha=0.5, label="slope 1 (diffusive)")
plt.loglog(t_ref, msd[0] * (t_ref / t_ref[0]) ** 2, "k:",  lw=1, alpha=0.5, label="slope 2 (ballistic)")
plt.xlabel("lag time  tau  (frames)"); plt.ylabel("MSD(tau)")
plt.title("log MSD vs log tau: simulated (points) vs true/analytical (lines)")
plt.grid(alpha=0.3, which="both"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

alpha_df = pd.DataFrame(alpha_rows)
print("Fitted (sim) vs true (theory) MSD exponents alpha:")
alpha_df

**Reading it**: `random_walk` should give alpha ~ 1 at both short and long
lags (a straight slope-1 line). `persistent` should give alpha close to 2 at
short lags (parallel to the ballistic guide line) and bend down toward alpha ~ 1
at long lags once the heading has decorrelated — the hallmark of a persistent
random walk. The crossover lag scales with the persistence time
~ `1 / (SIGMA_THETA**2 / 2)`; lower `SIGMA_THETA` pushes the crossover later and
keeps alpha near 2 longer.

## 3. Tracking methods

Each method takes the per-frame detection list and returns tracks as
`{track_id: [(frame, detection_index), ...]}`.

In [ ]:
def track_nearest_neighbour(dets, max_dist=MAX_DIST):
    '''Greedy nearest-neighbour frame-to-frame linking.'''
    tracks, active, next_id = {}, {}, 0
    for i in range(len(dets[0])):
        tracks[next_id] = [(0, i)]; active[i] = next_id; next_id += 1
    for t in range(len(dets) - 1):
        cost = cdist(dets[t], dets[t + 1])
        new_active, used = {}, set()
        for r in range(cost.shape[0]):
            j = int(np.argmin(cost[r]))
            if cost[r, j] <= max_dist and j not in used and r in active:
                tracks[active[r]].append((t + 1, j)); new_active[j] = active[r]; used.add(j)
        for j in range(len(dets[t + 1])):
            if j not in new_active:
                tracks[next_id] = [(t + 1, j)]; new_active[j] = next_id; next_id += 1
        active = new_active
    return tracks


def track_hungarian(dets, max_dist=MAX_DIST):
    '''Globally optimal frame-to-frame assignment (Hungarian / LAP).'''
    tracks, active, next_id = {}, {}, 0
    for i in range(len(dets[0])):
        tracks[next_id] = [(0, i)]; active[i] = next_id; next_id += 1
    for t in range(len(dets) - 1):
        cost = cdist(dets[t], dets[t + 1])
        rows, cols = linear_sum_assignment(cost)
        new_active = {}
        for r, c in zip(rows, cols):
            if cost[r, c] <= max_dist and r in active:
                tracks[active[r]].append((t + 1, c)); new_active[c] = active[r]
        for j in range(len(dets[t + 1])):
            if j not in new_active:
                tracks[next_id] = [(t + 1, j)]; new_active[j] = next_id; next_id += 1
        active = new_active
    return tracks


def _dets_to_dataframe(dets):
    rows = []
    for t, p in enumerate(dets):
        for i, (x, y) in enumerate(p):
            rows.append({"frame": t, "x": float(x), "y": float(y), "idx": i})
    return pd.DataFrame(rows)


def track_laptrack(dets, max_dist=MAX_DIST):
    '''LapTrack LAP tracker (ImportError if not installed).'''
    from laptrack import LapTrack
    df = _dets_to_dataframe(dets)
    lt = LapTrack(
        track_dist_metric="sqeuclidean", track_cost_cutoff=max_dist ** 2,
        gap_closing_dist_metric="sqeuclidean", gap_closing_cost_cutoff=max_dist ** 2,
        gap_closing_max_frame_count=2,
    )
    track_df, _, _ = lt.predict_dataframe(
        df, coordinate_cols=["x", "y"], frame_col="frame", only_coordinate_cols=False)
    track_df = track_df.reset_index()
    return {int(tid): [(int(r.frame), int(r.idx)) for r in g.itertuples()]
            for tid, g in track_df.groupby("track_id")}


def track_trackpy(dets, max_dist=MAX_DIST):
    '''trackpy Crocker-Grier linking (ImportError if not installed).'''
    import trackpy as tp
    tp.quiet()
    linked = tp.link(_dets_to_dataframe(dets), search_range=max_dist, memory=0)
    return {int(pid): [(int(r.frame), int(r.idx)) for r in g.itertuples()]
            for pid, g in linked.groupby("particle")}


ALL_METHODS = {
    "NearestNeighbour": track_nearest_neighbour,
    "Hungarian(LAP)":   track_hungarian,
    "LapTrack":         track_laptrack,
    "trackpy":          track_trackpy,
}

def available_methods():
    methods = {}
    for name, fn in ALL_METHODS.items():
        try:
            fn([np.zeros((2, 2)), np.ones((2, 2))])
            methods[name] = fn
        except ImportError:
            print(f"[skip] {name}: package not installed")
        except Exception as e:
            print(f"[warn] {name}: smoke test failed ({e}); keeping anyway")
            methods[name] = fn
    return methods

METHODS = available_methods()
print("Active methods:", list(METHODS))

## 4. Scoring against ground truth

A predicted link joins consecutive detections in a track; it is *correct* when
both belong to the same true cell. True links = `n_cells * (n_frames - 1)`.

In [ ]:
def score_tracks(tracks, labels, n_cells, n_frames=N_FRAMES):
    correct = pred_total = 0
    purities = []
    for dets in tracks.values():
        dets = sorted(dets)
        for (t1, i), (t2, j) in zip(dets, dets[1:]):
            if t2 == t1 + 1:
                pred_total += 1
                if labels[t1][i] == labels[t2][j]:
                    correct += 1
        ids = [labels[t][i] for (t, i) in dets]
        if ids:
            _, counts = np.unique(ids, return_counts=True)
            purities.append(counts.max() / len(ids))
    true_total = n_cells * (n_frames - 1)
    recall = correct / true_total if true_total else 0.0
    precision = correct / pred_total if pred_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"recall": recall, "precision": precision, "f1": f1,
            "track_purity": float(np.mean(purities)) if purities else 0.0,
            "n_pred_tracks": len(tracks)}

## 5. Run the benchmark (2 motions x 3 densities x 15 reps)

Same detections feed every method within a run.

In [ ]:
records = []
for motion in MOTIONS:
    for density in DENSITIES:
        for rep in range(N_REPEATS):
            s1 = int(rng_master.integers(0, 2**31 - 1))
            s2 = int(rng_master.integers(0, 2**31 - 1))
            pos = simulate_cells(density, motion, seed=s1)
            dets, labels = make_detections(pos, seed=s2)
            for name, fn in METHODS.items():
                try:
                    m = score_tracks(fn(dets, max_dist=MAX_DIST), labels, density)
                    m.update(method=name, motion=motion, density=density, repeat=rep,
                             ratio=difficulty_ratio(density))
                    records.append(m)
                except Exception as e:
                    print(f"[error] {name} {motion} n={density} rep={rep}: {e}")
        print(f"done {motion} n={density}")

results = pd.DataFrame.from_records(records)
print("Total runs:", len(results))
results.head()

## 6. Aggregate results

In [ ]:
summary = (results
    .groupby(["motion", "density", "method"])[["f1", "precision", "recall", "track_purity"]]
    .mean().round(3))
summary

In [ ]:
# Mean link-F1 tables, one per motion model (rows = method, cols = density)
for motion in MOTIONS:
    print(f"\n=== {motion}: mean link-F1 ===")
    print(results[results.motion == motion]
          .pivot_table(index="method", columns="density", values="f1").round(3))

## 7. F1 vs. difficulty ratio, per motion model

One panel per motion model; one line per method. The dashed line marks `r = 1`
(cell displacement ~ neighbour spacing).

In [ ]:
fig, axes = plt.subplots(1, len(MOTIONS), figsize=(7 * len(MOTIONS), 5), sharey=True)
if len(MOTIONS) == 1:
    axes = [axes]
for ax, motion in zip(axes, MOTIONS):
    sub = results[results.motion == motion]
    for name in METHODS:
        g = sub[sub.method == name].groupby("ratio")["f1"]
        ax.errorbar(g.mean().index, g.mean().values, yerr=g.std().values,
                    marker="o", capsize=4, label=name)
    ax.axvline(1.0, color="grey", ls="--", lw=1)
    for d in DENSITIES:
        ax.annotate(f"n={d}", (difficulty_ratio(d), 1.01), ha="center", fontsize=8)
    ax.set_xlabel("difficulty ratio  r = displacement / neighbour spacing")
    ax.set_title(f"{motion}")
    ax.set_ylim(0, 1.05); ax.grid(alpha=0.3); ax.legend()
axes[0].set_ylabel("link F1  (mean +/- std, N=%d)" % N_REPEATS)
plt.tight_layout(); plt.show()

## 8. Critical difficulty ratio per method

The ratio at which performance degrades is **not the same for every method**.
We estimate, for each (motion, method), the ratio `r` where the mean link-F1
first drops below a threshold (0.9 = "starting to fail", 0.5 = "half wrong"),
by linear interpolation between measured densities. Higher critical ratio =
more robust to crowding/fast motion.

Three densities give a coarse estimate; run the fine sweep in section 9 for a
smoother, more reliable value.

In [ ]:
def critical_ratio(df, method, motion, thresh):
    '''Interpolate the ratio where mean F1 crosses `thresh` (going down).'''
    g = (df[(df.method == method) & (df.motion == motion)]
         .groupby("ratio")["f1"].mean().sort_index())
    r, f = g.index.values, g.values
    if len(r) < 2:
        return np.nan
    for k in range(len(r) - 1):
        if f[k] >= thresh > f[k + 1]:          # crossing between k and k+1
            frac = (f[k] - thresh) / (f[k] - f[k + 1])
            return r[k] + frac * (r[k + 1] - r[k])
    if f[0] < thresh:
        return r[0]           # already below at the smallest ratio tested
    return np.nan             # never dropped below within tested range


def critical_table(df, thresholds=(0.9, 0.5)):
    rows = []
    for motion in MOTIONS:
        for name in METHODS:
            row = {"motion": motion, "method": name}
            for th in thresholds:
                row[f"r@F1={th}"] = round(critical_ratio(df, name, motion, th), 3)
            rows.append(row)
    return pd.DataFrame(rows)

crit = critical_table(results)
print("Critical difficulty ratio (higher = more robust; NaN = did not cross in range)")
crit

## 9. Optional fine sweep -> smooth F1(r) curves and better critical ratios

Three densities pin only three points on the `r` axis. This sweep uses many
densities (few reps each) for both motion models, giving continuous F1(r) curves
and more reliable critical ratios. Set `RUN_FINE_SWEEP = True` to run it
(a few minutes; includes high-density cases).

In [ ]:
RUN_FINE_SWEEP = False   # flip to True

if RUN_FINE_SWEEP:
    fine_densities = [3, 10, 25, 50, 100, 200, 350, 560]
    fine_repeats = 6
    frng = np.random.default_rng(BASE_SEED + 7)
    frecs = []
    for motion in MOTIONS:
        for density in fine_densities:
            for rep in range(fine_repeats):
                pos = simulate_cells(density, motion, seed=int(frng.integers(0, 2**31 - 1)))
                dets, labels = make_detections(pos, seed=int(frng.integers(0, 2**31 - 1)))
                for name, fn in METHODS.items():
                    try:
                        m = score_tracks(fn(dets, max_dist=MAX_DIST), labels, density)
                        m.update(method=name, motion=motion, density=density,
                                 ratio=difficulty_ratio(density))
                        frecs.append(m)
                    except Exception as e:
                        print(f"[error] {name} {motion} n={density}: {e}")
            print(f"fine done {motion} n={density}")

    fine = pd.DataFrame.from_records(frecs)

    fig, axes = plt.subplots(1, len(MOTIONS), figsize=(7 * len(MOTIONS), 5), sharey=True)
    if len(MOTIONS) == 1:
        axes = [axes]
    for ax, motion in zip(axes, MOTIONS):
        sub = fine[fine.motion == motion]
        for name in METHODS:
            g = sub[sub.method == name].groupby("ratio")["f1"].mean()
            ax.plot(g.index, g.values, marker="o", label=name)
        ax.axvline(1.0, color="grey", ls="--", lw=1)
        ax.set_xlabel("difficulty ratio  r"); ax.set_title(motion)
        ax.set_ylim(0, 1.05); ax.grid(alpha=0.3); ax.legend()
    axes[0].set_ylabel("mean link F1")
    plt.tight_layout(); plt.show()

    print("\nCritical ratios from fine sweep:")
    display(critical_table(fine))
else:
    print("RUN_FINE_SWEEP is False - skipping.")

## 10. Example trajectories (both motion models)

Ground truth vs. recovered tracks at the hardest density, side by side for each
motion model. Notice how much straighter the persistent tracks are.

In [ ]:
demo_density = DENSITIES[-1]
for motion in MOTIONS:
    pos = simulate_cells(demo_density, motion, seed=BASE_SEED + 999)
    dets, labels = make_detections(pos, seed=BASE_SEED + 1000)
    n_panels = 1 + len(METHODS)
    fig, axes = plt.subplots(1, n_panels, figsize=(4.0 * n_panels, 4.2))
    if n_panels == 1:
        axes = [axes]
    ax = axes[0]
    for c in range(demo_density):
        ax.plot(pos[:, c, 0], pos[:, c, 1], lw=0.6, alpha=0.6)
    ax.set_title(f"{motion}\nground truth (n={demo_density})")
    ax.set_xlim(0, GRID); ax.set_ylim(0, GRID); ax.set_aspect("equal")
    for ax, (name, fn) in zip(axes[1:], METHODS.items()):
        try:
            tracks = fn(dets, max_dist=MAX_DIST)
            for _, dd in tracks.items():
                dd = sorted(dd)
                xy = np.array([dets[t][i] for (t, i) in dd])
                if len(xy) > 1:
                    ax.plot(xy[:, 0], xy[:, 1], lw=0.6, alpha=0.6)
            m = score_tracks(tracks, labels, demo_density)
            ax.set_title(f"{name}\nF1={m['f1']:.2f}")
        except Exception as e:
            ax.set_title(f"{name}\n({type(e).__name__})")
        ax.set_xlim(0, GRID); ax.set_ylim(0, GRID); ax.set_aspect("equal")
    plt.tight_layout(); plt.show()

## 10b. Comparing the methods by recovered alpha (biophysical fidelity)

Link-F1 measures *linking* accuracy. But a common downstream goal is to recover
the **motion statistics** — here, the MSD exponent alpha. A tracker that makes ID
switches stitches pieces of *different* cells together, injecting spurious jumps
that inflate the MSD and bias alpha; fragmentation shortens tracks and shrinks the
usable lag range. So we ask: **does each method reproduce the true alpha?**

For each motion model and density we compute, on the *same* detections:

* **true alpha** — from ground-truth-labelled tracks, and
* **recovered alpha** — from each method's output tracks,

using an all-pairs time-averaged MSD estimator that tolerates gaps. The method
whose recovered alpha is closest to the true alpha best preserves the biology.

In [ ]:
def ground_truth_tracks(labels):
    '''Perfect tracks from the true labels: {cell_id: [(frame, det_index), ...]}.'''
    gt = {}
    for t, lab in enumerate(labels):
        for i, cid in enumerate(lab):
            gt.setdefault(int(cid), []).append((t, i))
    return gt


def msd_from_tracks(tracks, dets, max_lag):
    '''All-pairs time-averaged MSD from (possibly gappy/short) tracks.'''
    ssum = np.zeros(max_lag + 1)
    cnt  = np.zeros(max_lag + 1)
    for tr in tracks.values():
        pts = sorted(tr)
        if len(pts) < 2:
            continue
        frames = np.array([t for t, _ in pts])
        xy = np.array([dets[t][i] for t, i in pts], float)
        for a in range(len(pts)):
            dl = frames[a + 1:] - frames[a]
            valid = dl <= max_lag
            if not valid.any():
                continue
            d2 = np.sum((xy[a + 1:] - xy[a]) ** 2, axis=1)
            for lag, sq in zip(dl[valid], d2[valid]):
                ssum[lag] += sq
                cnt[lag] += 1
    taus = np.arange(1, max_lag + 1)
    msd = np.array([ssum[t] / cnt[t] if cnt[t] > 0 else np.nan for t in taus])
    return taus, msd


ALPHA_DENSITIES = list(DENSITIES)   # n = 5, 48, 480
ALPHA_REPS      = 5
ALPHA_MAXLAG    = 15            # bounded trajectories -> keep lags short
ALPHA_FIT       = (1, 5)       # short-lag window for alpha

alpha_recs = []
for motion in MOTIONS:
    for density in ALPHA_DENSITIES:
        acc = {"TRUE": []}
        acc.update({name: [] for name in METHODS})
        for rep in range(ALPHA_REPS):
            pos = simulate_cells(density, motion, seed=BASE_SEED + 500 + rep)
            dets, labels = make_detections(pos, seed=BASE_SEED + 900 + rep)
            _, msd_true = msd_from_tracks(ground_truth_tracks(labels), dets, ALPHA_MAXLAG)
            acc["TRUE"].append(msd_true)
            for name, fn in METHODS.items():
                try:
                    _, msd_m = msd_from_tracks(fn(dets, max_dist=MAX_DIST), dets, ALPHA_MAXLAG)
                    acc[name].append(msd_m)
                except Exception as e:
                    print(f"[error] {name} {motion} n={density}: {e}")
        taus = np.arange(1, ALPHA_MAXLAG + 1)
        a_true = fit_alpha(taus, np.nanmean(acc["TRUE"], axis=0), *ALPHA_FIT)[0]
        for name in METHODS:
            if acc[name]:
                a_m = fit_alpha(taus, np.nanmean(acc[name], axis=0), *ALPHA_FIT)[0]
                alpha_recs.append({"motion": motion, "density": density, "method": name,
                                   "alpha_true": round(a_true, 3),
                                   "alpha_recovered": round(a_m, 3),
                                   "alpha_error": round(abs(a_m - a_true), 3)})
    print(f"alpha recovery done: {motion}")

alpha_recovery = pd.DataFrame(alpha_recs)
print("Recovered vs true alpha (short-lag); alpha_error = |recovered - true|:")
alpha_recovery

In [ ]:
# Bar chart of |recovered alpha - true alpha| per method (lower = more faithful)
fig, axes = plt.subplots(1, len(MOTIONS), figsize=(7 * len(MOTIONS), 4.5), sharey=True)
if len(MOTIONS) == 1:
    axes = [axes]
methods = list(METHODS)
width = 0.8 / max(len(methods), 1)
xpos = np.arange(len(ALPHA_DENSITIES))
for ax, motion in zip(axes, MOTIONS):
    sub = alpha_recovery[alpha_recovery.motion == motion]
    for k, name in enumerate(methods):
        errs = [sub[(sub.method == name) & (sub.density == d)]["alpha_error"].mean()
                for d in ALPHA_DENSITIES]
        ax.bar(xpos + k * width, errs, width, label=name)
    ax.set_xticks(xpos + width * (len(methods) - 1) / 2)
    ax.set_xticklabels([f"n={d}" for d in ALPHA_DENSITIES])
    ax.set_title(motion); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel("|recovered alpha - true alpha|")
plt.suptitle("Motion-statistics fidelity: how far each method's alpha is from truth")
plt.tight_layout(); plt.show()

## 10c. Comparing the methods by recovered average speed

Average speed is the other headline motion statistic. We estimate it as the mean
frame-to-frame step length divided by the time interval (5 min), so units are
grid-units / min. As with alpha, tracking errors bias it — but here the dominant
effect is that linkers preferentially connect the *nearest* available detection,
which are the *shorter* steps, so a crowded tracker tends to **under-estimate**
average speed, and the bias grows with density. (Fragmentation reinforces this:
the links that survive are the easy, short ones.)

The two models have **different true speeds even though their RMS displacement is
matched** (that match was done in the *squared* sense):

* random walk — step length is Rayleigh-distributed, mean `sigma*sqrt(pi/2)`.
* persistent walk — fixed step length `sigma*sqrt(2)` (always larger).

We compare, at densities **n = 5, 48, 480**, the true speed (from
ground-truth-labelled tracks) with each method's recovered speed, and also show
the analytical reference.

In [ ]:
def speed_from_tracks(tracks, dets, dt_min=DT_MIN):
    '''Mean frame-to-frame step length, and speed per minute.'''
    disps = []
    for tr in tracks.values():
        pts = sorted(tr)
        for (t1, i), (t2, j) in zip(pts, pts[1:]):
            if t2 == t1 + 1:
                d = dets[t2][j] - dets[t1][i]
                disps.append(float(np.hypot(d[0], d[1])))
    if not disps:
        return np.nan, np.nan
    step = float(np.mean(disps))
    return step, step / dt_min


def true_speed_theory(motion, sigma=SIGMA_STEP, dt_min=DT_MIN):
    '''Analytical mean per-frame step length / dt (ignores loc noise & walls).'''
    if motion == "random_walk":
        step = sigma * np.sqrt(np.pi / 2.0)      # mean of a Rayleigh(sigma)
    elif motion == "persistent":
        step = sigma * np.sqrt(2.0)              # fixed step length
    else:
        raise ValueError(motion)
    return step, step / dt_min


SPEED_DENSITIES = list(DENSITIES)   # n = 5, 48, 480
SPEED_REPS      = 5

speed_recs = []
for motion in MOTIONS:
    _, v_theory = true_speed_theory(motion)
    for density in SPEED_DENSITIES:
        acc = {"TRUE": []}
        acc.update({name: [] for name in METHODS})
        for rep in range(SPEED_REPS):
            pos = simulate_cells(density, motion, seed=BASE_SEED + 300 + rep)
            dets, labels = make_detections(pos, seed=BASE_SEED + 700 + rep)
            acc["TRUE"].append(speed_from_tracks(ground_truth_tracks(labels), dets)[1])
            for name, fn in METHODS.items():
                try:
                    acc[name].append(speed_from_tracks(fn(dets, max_dist=MAX_DIST), dets)[1])
                except Exception as e:
                    print(f"[error] {name} {motion} n={density}: {e}")
        v_true = float(np.nanmean(acc["TRUE"]))
        for name in METHODS:
            if acc[name]:
                v_m = float(np.nanmean(acc[name]))
                speed_recs.append({"motion": motion, "density": density, "method": name,
                                   "speed_theory": round(v_theory, 4),
                                   "speed_true": round(v_true, 4),
                                   "speed_recovered": round(v_m, 4),
                                   "pct_error": round(100 * (v_m - v_true) / v_true, 1)})
    print(f"speed recovery done: {motion}")

speed_recovery = pd.DataFrame(speed_recs)
print("Average speed (grid-units/min). pct_error = 100*(recovered - true)/true:")
speed_recovery

In [ ]:
# Recovered vs true average speed, per motion model
fig, axes = plt.subplots(1, len(MOTIONS), figsize=(7 * len(MOTIONS), 4.5), sharey=True)
if len(MOTIONS) == 1:
    axes = [axes]
methods = list(METHODS)
width = 0.8 / (len(methods) + 1)
xpos = np.arange(len(SPEED_DENSITIES))
for ax, motion in zip(axes, MOTIONS):
    sub = speed_recovery[speed_recovery.motion == motion]
    # true speed bars first
    v_true = [sub[sub.density == d]["speed_true"].iloc[0] for d in SPEED_DENSITIES]
    ax.bar(xpos, v_true, width, label="TRUE", color="0.3")
    for k, name in enumerate(methods, start=1):
        vals = [sub[(sub.method == name) & (sub.density == d)]["speed_recovered"].mean()
                for d in SPEED_DENSITIES]
        ax.bar(xpos + k * width, vals, width, label=name)
    _, v_th = true_speed_theory(motion)
    ax.axhline(v_th, color="red", ls="--", lw=1, label="analytical")
    ax.set_xticks(xpos + width * len(methods) / 2)
    ax.set_xticklabels([f"n={d}" for d in SPEED_DENSITIES])
    ax.set_title(motion); ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel("average speed (units / min)")
plt.suptitle("Average speed: true vs. recovered by each method (under-estimation grows with density)")
plt.tight_layout(); plt.show()

## 11. Notes

* **Comparability** — both motion models share the same RMS per-frame
  displacement, so the difficulty ratio `r = displacement / neighbour-spacing`
  means the same thing in both. Only the *character* of motion differs.
* **Why the critical ratio differs by method** — greedy nearest-neighbour
  commits to the closest detection frame-by-frame, so it starts making ID
  errors at a *lower* `r` than the global LAP methods (Hungarian, LapTrack),
  which resolve all competing assignments jointly. The `r@F1=0.5` column in
  section 8 quantifies this gap.
* **Random walk vs. persistent** — persistence changes the critical ratio, but
  the sign depends on your parameters. Straight runs make a single cell's next
  position more predictable (easier), yet coherent streaming can push many cells
  through the same gating region together (harder in crowds). Read the actual
  direction off the two panels in section 7 and the table in section 8 rather
  than assuming it; `SIGMA_THETA` controls how ballistic the motion is.
* **True vs. fitted alpha** — section 2b overlays the exact analytical MSD
  (RW: `2 sigma^2 tau`, alpha = 1; PRW: discrete Furth formula, alpha 2 -> 1),
  so the fitted alpha is validated against the *true* value, not just asserted.
* **Recovered alpha is a second way to rank methods** — section 10b shows that
  tracking errors distort the motion statistics: at high density the persistent
  cells' true short-lag alpha (~1.9) collapses toward ~1 in the recovered tracks,
  i.e. **bad tracking makes ballistic cells look diffusive**. `alpha_error` there
  ranks methods by biophysical fidelity, which can differ from the link-F1 ranking.
* **Recovered speed (section 10c)** — because linkers prefer the nearest (hence
  shorter) detection and keep the easy short links, methods tend to
  *under-estimate* average speed, and the bias grows sharply with density
  (near-zero at n=5, tens of percent at n=480). The two models have different true
  speeds (`sigma*sqrt(pi/2)` for RW vs `sigma*sqrt(2)` for PRW) even though their
  squared displacement was matched.
* **Tuning** — `SIGMA_STEP` (speed), `SIGMA_THETA` (persistence; smaller = more
  ballistic), and `MAX_DIST` (gating) are the main knobs. Note that a fixed
  `MAX_DIST` itself caps performance at high density; scale it with
  `nn_spacing(n)` if you want `r` to be the sole driver.
